In [3]:
import os
import pandas as pd
from google.cloud import bigquery

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '/Users/kasperhansen/Desktop/Gdelt/Json/keen-diode-493214-v8-d8b69dbc0a80.json'

client = bigquery.Client(project='keen-diode-493214-v8')

query = """
WITH parsed AS (
  SELECT 
    SPLIT(SPLIT(V2Locations, ';')[OFFSET(0)], '#')[OFFSET(1)] as Land,
    SPLIT(TRIM(SPLIT(V2Themes, ';')[OFFSET(0)]), ',')[OFFSET(0)] as Emne,
    SPLIT(V2Persons, ';')[OFFSET(0)] as Person,
    SPLIT(V2Organizations, ';')[OFFSET(0)] as Organisation,
    SPLIT(V2Counts, ';')[OFFSET(0)] as Counts_Raa,
    DocumentIdentifier,
    DATE
  FROM `gdelt-bq.gdeltv2.gkg_partitioned`
  WHERE _PARTITIONDATE = CURRENT_DATE()
    AND V2Themes LIKE '%ECON_%'
    AND V2Locations IS NOT NULL
)
SELECT 
    Land,
    Emne,
    Person,
    Organisation,
    Counts_Raa,
    COUNT(DISTINCT DocumentIdentifier) as Antal_Artikler
FROM parsed
WHERE Land IS NOT NULL AND Land != ''
GROUP BY Land, Emne, Person, Organisation, Counts_Raa
ORDER BY Antal_Artikler DESC
LIMIT 100
"""

query_job = client.query(query)
df = query_job.to_dataframe(create_bqstorage_client=False)

print("=== Økonomisk Politik - Idag ===")
print(df)
print(f"\nI alt {len(df)} rækker")

=== Økonomisk Politik - Idag ===
                          Land                        Emne Person  \
0                         Iran              TAX_ECON_PRICE   None   
1                         Chad              GENERAL_HEALTH   None   
2                         Chad              TAX_ECON_PRICE   None   
3                      Germany              TAX_ECON_PRICE   None   
4   Guan Cun, Guangdong, China         TAX_FNCACT_MERCHANT   None   
..                         ...                         ...    ...   
95                     Italian        EPU_ECONOMY_HISTORIC   None   
96                       Spain                 EPU_ECONOMY   None   
97                      France        TAX_ETHNICITY_FRENCH   None   
98               United States      CRISISLEX_CRISISLEXREC   None   
99                      France  UNGP_FORESTS_RIVERS_OCEANS   None   

                         Organisation Counts_Raa  Antal_Artikler  
0                                None       None              29  
1   

In [ ]:
query_events = """
WITH e AS (
  SELECT 
    g.Land,
    g.Emne,
    e.EventCode,
    e.Actor1CountryCode,
    e.Actor2CountryCode,
    e.NumArticles,
    e.NumMentions,
    e.Day
  FROM (
    SELECT 
      SPLIT(SPLIT(V2Locations, ';')[OFFSET(0)], '#')[OFFSET(1)] as Land,
      SPLIT(TRIM(SPLIT(V2Themes, ';')[OFFSET(0)]), ',')[OFFSET(0)] as Emne,
      DocumentIdentifier
    FROM `gdelt-bq.gdeltv2.gkg_partitioned`
    WHERE _PARTITIONDATE = CURRENT_DATE()
      AND V2Themes LIKE '%ECON_%'
  ) g
  JOIN `gdelt-bq.gdeltv2.events_partitioned` e
    ON g.DocumentIdentifier = e.GKGRecord
  WHERE e._PARTITIONDATE = CURRENT_DATE()
)
SELECT 
    Land,
    Emne,
    Actor1CountryCode as Land1,
    Actor2CountryCode as Land2,
    EventCode,
    SUM(CAST(NumMentions AS INT64)) as Samlet_Mentions,
    COUNT(*) as Antal_Begivenheder
FROM e
GROUP BY Land, Emne, Actor1CountryCode, Actor2CountryCode, EventCode
ORDER BY Samlet_Mentions DESC
LIMIT 50
"""

try:
    query_job = client.query(query_events)
    df_events = query_job.to_dataframe(create_bqstorage_client=False)
    print("\n=== Begivenheder Koblet til Økonomisk Politik ===")
    print(df_events)
except Exception as e:
    print(f"Kunne ikke hente events: {e}")